# 07. 자동화와 종료 상태 기반 테스트


## Goal

작은 CLI 스크립트를 만들고 성공·실패 경로를 자동으로 검증합니다.


## Setup


In [ ]:
from pathlib import Path
import os
import shutil
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="bash-book-07-"))
os.environ["BASH_LAB_DIR"] = str(lab_dir)
print(f"새 임시 실습 디렉터리: {lab_dir}")


## Steps

### 1. 검증 가능한 CLI 작성


In [ ]:
%%bash
set -euo pipefail
cat > "$BASH_LAB_DIR/count-lines.sh" <<'BASH'
#!/usr/bin/env bash
set -euo pipefail
if [[ $# -ne 1 ]]; then
  printf 'Usage: %s <readable-file>\n' "$0" >&2
  exit 64
fi
input=$1
if [[ ! -f $input || ! -r $input ]]; then
  printf 'not a readable file: %s\n' "$input" >&2
  exit 66
fi
wc -l < "$input" | tr -d ' '
BASH
chmod 700 "$BASH_LAB_DIR/count-lines.sh"
printf 'one\ntwo\nthree\n' > "$BASH_LAB_DIR/input.txt"


### 2. 성공 경로 테스트


In [ ]:
%%bash
set -euo pipefail
actual=$("$BASH_LAB_DIR/count-lines.sh" "$BASH_LAB_DIR/input.txt")
[[ $actual == 3 ]]
printf 'success test passed: %s lines\n' "$actual"


### 3. 실패 경로 테스트


In [ ]:
%%bash
set -euo pipefail
if "$BASH_LAB_DIR/count-lines.sh" "$BASH_LAB_DIR/missing.txt"     >"$BASH_LAB_DIR/out.txt" 2>"$BASH_LAB_DIR/err.txt"; then
  printf 'missing-file test failed\n' >&2
  exit 1
else
  status=$?
fi
[[ $status -eq 66 ]]
grep -q 'not a readable file' "$BASH_LAB_DIR/err.txt"
printf 'failure test passed: status=%s\n' "$status"


### 4. 선택적 정적 분석


In [ ]:
%%bash
set -euo pipefail
if command -v shellcheck >/dev/null 2>&1; then
  shellcheck "$BASH_LAB_DIR/count-lines.sh"
  printf 'ShellCheck passed\n'
else
  printf 'ShellCheck가 없어 실행을 건너뜁니다. Ubuntu: sudo apt install shellcheck\n'
fi


## Checks

- 정상 입력에서 `3`을 출력하고 종료 상태 `0`을 반환하는가?
- 없는 파일에서 종료 상태 `66`과 오류 메시지를 반환하는가?
- 테스트가 출력 문자열뿐 아니라 종료 상태도 검증하는가?


## Next Steps

지금까지의 패턴을 결합해 로컬 triage 수집기를 만듭니다.


In [ ]:
import shutil
from pathlib import Path
import os

lab_dir = Path(os.environ["BASH_LAB_DIR"])
shutil.rmtree(lab_dir, ignore_errors=True)
print(f"정리 완료: {lab_dir}")
